# 🥈 Silver Layer - Data Quality & Transformation

## Propósito
Transformación, limpieza y consolidación de datos con garantía de calidad.

## Arquitectura Medallion - Silver
La capa Silver contiene:
* **Datos limpios** con tipos correctos y formatos estandarizados
* **Validaciones de calidad** aplicadas (nulls, rangos, consistencia)
* **Deduplicación** por primary key (Nit)
* **Estandarización** de valores categóricos
* **Consolidación** de múltiples fuentes en entidades de negocio

## Tablas Silver Creadas
### Tablas Individuales Limpias:
1. `workspace.churn_silver.asociados_demographics` - Demográficos limpios con edad calculada
2. `workspace.churn_silver.retiros_aportes` - Retiros con fechas parseadas
3. `workspace.churn_silver.productos_ahorros` - Productos validados
4. `workspace.churn_silver.operaciones_transaccionales` - Métricas transaccionales
5. `workspace.churn_silver.interacciones_canal` - Canales estandarizados

### Tabla Consolidada (360° del Cliente):
6. **`workspace.churn_silver.clientes_consolidado`** - Vista única del cliente con todas las dimensiones

---
**Layer:** Silver (Curated)  
**Última actualización:** 2026-05-17  
**Transformaciones:** Limpieza + Validación + Consolidación

In [0]:
%run ./00_config

In [0]:
# ================================================================================
# UTILITY FUNCTIONS - SILVER TRANSFORMATION
# ================================================================================

from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, desc

def clean_string_column(df: DataFrame, column_name: str, uppercase: bool = False) -> DataFrame:
    """
    Limpia una columna string: trim espacios, remover nulls vacíos, opcional uppercase.
    """
    df = df.withColumn(column_name, trim(col(column_name)))
    df = df.withColumn(column_name, when(col(column_name) == "", None).otherwise(col(column_name)))
    
    if uppercase:
        df = df.withColumn(column_name, upper(col(column_name)))
    
    return df


def deduplicate_by_key(df: DataFrame, key_column: str, order_by_column: str = None) -> DataFrame:
    """
    Deduplica un DataFrame por clave, conservando el último registro según order_by.
    
    Args:
        df: DataFrame de entrada
        key_column: Columna clave para deduplicación (ej: 'Nit')
        order_by_column: Columna para ordenar (default: timestamp_ingestion descendente)
    
    Returns:
        DataFrame deduplicado
    """
    if order_by_column is None:
        order_by_column = "timestamp_ingestion"
    
    window = Window.partitionBy(key_column).orderBy(desc(order_by_column))
    
    df_dedup = df.withColumn("row_num", row_number().over(window)) \
        .filter(col("row_num") == 1) \
        .drop("row_num")
    
    return df_dedup


def validate_not_null(df: DataFrame, columns: list, table_name: str) -> DataFrame:
    """
    Valida que columnas críticas no tengan nulls. Filtra registros inválidos.
    """
    initial_count = df.count()
    
    for column in columns:
        df = df.filter(col(column).isNotNull())
    
    final_count = df.count()
    removed = initial_count - final_count
    
    if removed > 0:
        logger.warning(f"⚠ {table_name}: {removed} registros removidos por nulls en {columns}")
    
    return df


print("✓ Funciones de transformación Silver cargadas")
print("  - clean_string_column()")
print("  - deduplicate_by_key()")
print("  - validate_not_null()")

In [0]:
# ================================================================================
# SILVER 1: DEMOGRAPHICS - demo_asociados
# ================================================================================

print("\n" + "="*80)
print("🔄 SILVER 1/5: Transformando Demográficos")
print("="*80)

# Leer tabla Bronze
df_demo_bronze = spark.table(SOURCE_TABLES_CONFIG["demo_asociados"]["bronze_table"])

logger.info(f"Registros leídos de Bronze: {df_demo_bronze.count():,}")

# Transformaciones
df_demo_silver = df_demo_bronze

# 1. Validar Nit no nulo
df_demo_silver = validate_not_null(df_demo_silver, ["Nit"], "demo_asociados")

# 2. Parsear Fechnacimiento a timestamp y calcular edad
df_demo_silver = df_demo_silver.withColumn(
    "fecha_nacimiento",
    to_timestamp(col("Fechnacimiento"), "yyyy-MM-dd HH:mm:ss")
)

df_demo_silver = df_demo_silver.withColumn(
    "edad",
    (datediff(current_date(), col("fecha_nacimiento")) / 365.25).cast("int")
)

# 3. Limpiar y estandarizar columnas categóricas
df_demo_silver = clean_string_column(df_demo_silver, "Sexo")
df_demo_silver = clean_string_column(df_demo_silver, "Nivel_ingresos")
df_demo_silver = clean_string_column(df_demo_silver, "Participacion_social")
df_demo_silver = clean_string_column(df_demo_silver, "Uso_creditos")

# 4. Deduplicación por Nit
df_demo_silver = deduplicate_by_key(df_demo_silver, "Nit")

# 5. Seleccionar columnas finales
df_demo_silver = df_demo_silver.select(
    "Nit",
    "Sexo",
    "fecha_nacimiento",
    "edad",
    "Permanencia",
    "Antasociado",
    "Nivel_ingresos",
    "Participacion_social",
    "Uso_creditos",
    "timestamp_ingestion",
    "load_date"
)

# 6. Escribir en Silver
silver_table = SOURCE_TABLES_CONFIG["demo_asociados"]["silver_table"]

df_demo_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .option("delta.autoOptimize.optimizeWrite", "true") \
    .saveAsTable(silver_table)

logger.info(f"✓ Tabla Silver creada: {silver_table}")

# 7. Optimizar
if get_execution_params()["run_optimization"]:
    optimize_delta_table(silver_table, ["Nit"])

log_ingestion_stats(df_demo_silver, silver_table, "SILVER_TRANSFORM")

print(f"✅ Silver 1/5 completado: {silver_table}")

In [0]:
# ================================================================================
# SILVER 2: RETIROS - retiro_aportes
# ================================================================================

print("\n" + "="*80)
print("🔄 SILVER 2/5: Transformando Retiros de Aportes")
print("="*80)

df_retiros_bronze = spark.table(SOURCE_TABLES_CONFIG["retiro_aportes"]["bronze_table"])

logger.info(f"Registros leídos de Bronze: {df_retiros_bronze.count():,}")

# Transformaciones
df_retiros_silver = df_retiros_bronze

# 1. Validar Nit no nulo
df_retiros_silver = validate_not_null(df_retiros_silver, ["Nit"], "retiro_aportes")

# 2. Parsear fechas
df_retiros_silver = df_retiros_silver.withColumn(
    "fecha_solicitud",
    to_date(col("Fechasolicitud"), "yyyy-MM-dd")
)

df_retiros_silver = df_retiros_silver.withColumn(
    "fecha_entrega",
    to_date(col("Fechaentrega"), "yyyy-MM-dd")
)

# 3. Calcular días de procesamiento
df_retiros_silver = df_retiros_silver.withColumn(
    "dias_procesamiento",
    datediff(col("fecha_entrega"), col("fecha_solicitud"))
)

# 4. Validar montos positivos
df_retiros_silver = df_retiros_silver.filter(col("Valoraportes") > 0)

# 5. Limpiar categorías
df_retiros_silver = clean_string_column(df_retiros_silver, "Motivo")
df_retiros_silver = clean_string_column(df_retiros_silver, "Castigo_cartera")
df_retiros_silver = clean_string_column(df_retiros_silver, "Tendencia_saldo")

# 6. Deduplicación por Nit (conservar el retiro más reciente)
window_retiros = Window.partitionBy("Nit").orderBy(desc("fecha_solicitud"))
df_retiros_silver = df_retiros_silver.withColumn("row_num", row_number().over(window_retiros)) \
    .filter(col("row_num") == 1) \
    .drop("row_num")

# 7. Seleccionar columnas finales
df_retiros_silver = df_retiros_silver.select(
    "Nit",
    "fecha_solicitud",
    "fecha_entrega",
    "dias_procesamiento",
    "Valoraportes",
    "Motivo",
    "Oficina",
    "Castigo_cartera",
    "Tiempo_asociado",
    "Frecuencia_retiros_previos",
    "Tendencia_saldo",
    "timestamp_ingestion",
    "load_date"
)

# 8. Escribir en Silver
silver_table = SOURCE_TABLES_CONFIG["retiro_aportes"]["silver_table"]

df_retiros_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .option("delta.autoOptimize.optimizeWrite", "true") \
    .saveAsTable(silver_table)

if get_execution_params()["run_optimization"]:
    optimize_delta_table(silver_table, ["Nit"])

log_ingestion_stats(df_retiros_silver, silver_table, "SILVER_TRANSFORM")

print(f"✅ Silver 2/5 completado: {silver_table}")

In [0]:
# ================================================================================
# SILVER 3: PRODUCTOS AHORRO - plano_ahorros
# ================================================================================

print("\n" + "="*80)
print("🔄 SILVER 3/5: Transformando Productos de Ahorro")
print("="*80)

df_ahorros_bronze = spark.table(SOURCE_TABLES_CONFIG["plano_ahorros"]["bronze_table"])

logger.info(f"Registros leídos de Bronze: {df_ahorros_bronze.count():,}")

# Transformaciones
df_ahorros_silver = df_ahorros_bronze

# 1. Validar Nit no nulo
df_ahorros_silver = validate_not_null(df_ahorros_silver, ["Nit"], "plano_ahorros")

# 2. Validar saldos no negativos
df_ahorros_silver = df_ahorros_silver.filter(col("Saldo_productos") >= 0)

# 3. Deduplicación por Nit
df_ahorros_silver = deduplicate_by_key(df_ahorros_silver, "Nit")

# 4. Seleccionar columnas finales
df_ahorros_silver = df_ahorros_silver.select(
    "Nit",
    "Productos_ahorro",
    "Saldo_productos",
    "Variacion_saldo_6m",
    "Diversificacion_productos",
    "timestamp_ingestion",
    "load_date"
)

# 5. Escribir en Silver
silver_table = SOURCE_TABLES_CONFIG["plano_ahorros"]["silver_table"]

df_ahorros_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .option("delta.autoOptimize.optimizeWrite", "true") \
    .saveAsTable(silver_table)

if get_execution_params()["run_optimization"]:
    optimize_delta_table(silver_table, ["Nit"])

log_ingestion_stats(df_ahorros_silver, silver_table, "SILVER_TRANSFORM")

print(f"✅ Silver 3/5 completado: {silver_table}")

In [0]:
# ================================================================================
# SILVER 4: OPERACIONES TRANSACCIONALES - detallado_operaciones
# ================================================================================

print("\n" + "="*80)
print("🔄 SILVER 4/5: Transformando Operaciones Transaccionales")
print("="*80)

df_ops_bronze = spark.table(SOURCE_TABLES_CONFIG["detallado_operaciones"]["bronze_table"])

logger.info(f"Registros leídos de Bronze: {df_ops_bronze.count():,}")

# Transformaciones
df_ops_silver = df_ops_bronze

# 1. Validar Nit no nulo
df_ops_silver = validate_not_null(df_ops_silver, ["Nit"], "detallado_operaciones")

# 2. Validar métricas no negativas
df_ops_silver = df_ops_silver.filter(
    (col("Numero_Tx") >= 0) & 
    (col("Frecuencia_tx_mensual") >= 0) &
    (col("Ultima_tx_dias") >= 0)
)

# 3. Deduplicación por Nit
df_ops_silver = deduplicate_by_key(df_ops_silver, "Nit")

# 4. Seleccionar columnas finales
df_ops_silver = df_ops_silver.select(
    "Nit",
    "Numero_Tx",
    "Frecuencia_tx_mensual",
    "Ultima_tx_dias",
    "timestamp_ingestion",
    "load_date"
)

# 5. Escribir en Silver
silver_table = SOURCE_TABLES_CONFIG["detallado_operaciones"]["silver_table"]

df_ops_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .option("delta.autoOptimize.optimizeWrite", "true") \
    .saveAsTable(silver_table)

if get_execution_params()["run_optimization"]:
    optimize_delta_table(silver_table, ["Nit"])

log_ingestion_stats(df_ops_silver, silver_table, "SILVER_TRANSFORM")

print(f"✅ Silver 4/5 completado: {silver_table}")

In [0]:
# ================================================================================
# SILVER 5: INTERACCIONES POR CANAL - publiturno
# ================================================================================

print("\n" + "="*80)
print("🔄 SILVER 5/5: Transformando Interacciones por Canal")
print("="*80)

df_publi_bronze = spark.table(SOURCE_TABLES_CONFIG["publiturno"]["bronze_table"])

logger.info(f"Registros leídos de Bronze: {df_publi_bronze.count():,}")

# Transformaciones
df_publi_silver = df_publi_bronze

# 1. Validar Nit no nulo
df_publi_silver = validate_not_null(df_publi_silver, ["Nit"], "publiturno")

# 2. Limpiar y estandarizar canales
df_publi_silver = clean_string_column(df_publi_silver, "Llegada_agencia")
df_publi_silver = clean_string_column(df_publi_silver, "Interaccion_linea")
df_publi_silver = clean_string_column(df_publi_silver, "Canal_preferido")

# 3. Deduplicación por Nit
df_publi_silver = deduplicate_by_key(df_publi_silver, "Nit")

# 4. Seleccionar columnas finales
df_publi_silver = df_publi_silver.select(
    "Nit",
    "Llegada_agencia",
    "Interaccion_linea",
    "Canal_preferido",
    "Tiempo_desde_ultima_interaccion",
    "timestamp_ingestion",
    "load_date"
)

# 5. Escribir en Silver
silver_table = SOURCE_TABLES_CONFIG["publiturno"]["silver_table"]

df_publi_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .option("delta.autoOptimize.optimizeWrite", "true") \
    .saveAsTable(silver_table)

if get_execution_params()["run_optimization"]:
    optimize_delta_table(silver_table, ["Nit"])

log_ingestion_stats(df_publi_silver, silver_table, "SILVER_TRANSFORM")

print(f"✅ Silver 5/5 completado: {silver_table}")

In [0]:
# ================================================================================
# SILVER CONSOLIDADO: VISTA 360° DEL CLIENTE
# ================================================================================

print("\n" + "="*80)
print("🌟 CREANDO TABLA CONSOLIDADA - Vista 360° del Cliente")
print("="*80)

# Leer tablas Silver individuales
df_demo = spark.table(SOURCE_TABLES_CONFIG["demo_asociados"]["silver_table"])
df_retiros = spark.table(SOURCE_TABLES_CONFIG["retiro_aportes"]["silver_table"])
df_ahorros = spark.table(SOURCE_TABLES_CONFIG["plano_ahorros"]["silver_table"])
df_ops = spark.table(SOURCE_TABLES_CONFIG["detallado_operaciones"]["silver_table"])
df_publi = spark.table(SOURCE_TABLES_CONFIG["publiturno"]["silver_table"])

logger.info("Tablas Silver individuales leídas")

# Crear vista consolidada con LEFT JOINs desde demo (base)
df_consolidado = df_demo.alias("demo") \
    .join(
        df_retiros.select(
            "Nit",
            col("fecha_solicitud").alias("retiro_fecha_solicitud"),
            col("fecha_entrega").alias("retiro_fecha_entrega"),
            col("dias_procesamiento").alias("retiro_dias_procesamiento"),
            col("Valoraportes").alias("retiro_valor"),
            col("Motivo").alias("retiro_motivo"),
            col("Oficina").alias("retiro_oficina"),
            col("Castigo_cartera").alias("tiene_castigo_cartera"),
            col("Frecuencia_retiros_previos").alias("retiro_frecuencia_previa"),
            col("Tendencia_saldo").alias("retiro_tendencia_saldo")
        ).alias("retiros"),
        "Nit",
        "left"
    ) \
    .join(
        df_ahorros.select(
            "Nit",
            col("Productos_ahorro").alias("num_productos_ahorro"),
            col("Saldo_productos").alias("saldo_total_productos"),
            col("Variacion_saldo_6m").alias("variacion_saldo_6m"),
            col("Diversificacion_productos").alias("nivel_diversificacion")
        ).alias("ahorros"),
        "Nit",
        "left"
    ) \
    .join(
        df_ops.select(
            "Nit",
            col("Numero_Tx").alias("numero_transacciones"),
            col("Frecuencia_tx_mensual").alias("frecuencia_tx_mensual"),
            col("Ultima_tx_dias").alias("dias_desde_ultima_tx")
        ).alias("ops"),
        "Nit",
        "left"
    ) \
    .join(
        df_publi.select(
            "Nit",
            col("Llegada_agencia").alias("visita_agencia"),
            col("Interaccion_linea").alias("usa_linea"),
            col("Canal_preferido").alias("canal_preferido"),
            col("Tiempo_desde_ultima_interaccion").alias("dias_desde_ultima_interaccion")
        ).alias("publi"),
        "Nit",
        "left"
    )

# Seleccionar todas las columnas con nombres claros
df_consolidado = df_consolidado.select(
    # Demográficos
    col("demo.Nit").alias("nit"),
    col("demo.Sexo").alias("sexo"),
    col("demo.fecha_nacimiento").alias("fecha_nacimiento"),
    col("demo.edad").alias("edad"),
    col("demo.Permanencia").alias("permanencia_meses"),
    col("demo.Antasociado").alias("antiguedad_meses"),
    col("demo.Nivel_ingresos").alias("nivel_ingresos"),
    col("demo.Participacion_social").alias("participacion_social"),
    col("demo.Uso_creditos").alias("uso_creditos"),
    
    # Retiros
    col("retiro_fecha_solicitud"),
    col("retiro_fecha_entrega"),
    col("retiro_dias_procesamiento"),
    col("retiro_valor"),
    col("retiro_motivo"),
    col("retiro_oficina"),
    col("tiene_castigo_cartera"),
    col("retiro_frecuencia_previa"),
    col("retiro_tendencia_saldo"),
    
    # Ahorros
    col("num_productos_ahorro"),
    col("saldo_total_productos"),
    col("variacion_saldo_6m"),
    col("nivel_diversificacion"),
    
    # Operaciones
    col("numero_transacciones"),
    col("frecuencia_tx_mensual"),
    col("dias_desde_ultima_tx"),
    
    # Interacciones
    col("visita_agencia"),
    col("usa_linea"),
    col("canal_preferido"),
    col("dias_desde_ultima_interaccion"),
    
    # Metadata
    col("demo.timestamp_ingestion").alias("timestamp_ingestion"),
    col("demo.load_date").alias("load_date")
)

logger.info(f"Vista consolidada creada: {df_consolidado.count():,} clientes")

# Escribir tabla consolidada
consolidado_table = f"{SILVER_SCHEMA}.clientes_consolidado"

df_consolidado.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .option("delta.autoOptimize.optimizeWrite", "true") \
    .saveAsTable(consolidado_table)

logger.info(f"✓ Tabla consolidada creada: {consolidado_table}")

# Optimizar
if get_execution_params()["run_optimization"]:
    optimize_delta_table(consolidado_table, ["nit"])

log_ingestion_stats(df_consolidado, consolidado_table, "SILVER_CONSOLIDATION")

print(f"\n✅ Tabla consolidada Silver creada: {consolidado_table}")
print(f"   • {df_consolidado.count():,} clientes con vista 360°")
print(f"   • {len(df_consolidado.columns)} columnas totales")

In [0]:
# ================================================================================
# RESUMEN DE TRANSFORMACIÓN SILVER
# ================================================================================

print("\n" + "="*80)
print("📊 RESUMEN DE TRANSFORMACIÓN - CAPA SILVER")
print("="*80)

# Listar todas las tablas Silver creadas
silver_tables = [
    config['silver_table'] 
    for config in SOURCE_TABLES_CONFIG.values()
]
silver_tables.append(f"{SILVER_SCHEMA}.clientes_consolidado")

print(f"\n✅ Total de tablas Silver: {len(silver_tables)}\n")

# Crear resumen
summary_data = []

for table_name, config in SOURCE_TABLES_CONFIG.items():
    silver_table = config['silver_table']
    df = spark.table(silver_table)
    
    summary_data.append({
        "Tabla_Origen": table_name,
        "Tabla_Silver": silver_table,
        "Registros": df.count(),
        "Columnas": len(df.columns)
    })

# Agregar consolidado
df_consolidado = spark.table(f"{SILVER_SCHEMA}.clientes_consolidado")
summary_data.append({
    "Tabla_Origen": "[CONSOLIDADO]",
    "Tabla_Silver": f"{SILVER_SCHEMA}.clientes_consolidado",
    "Registros": df_consolidado.count(),
    "Columnas": len(df_consolidado.columns)
})

df_summary = spark.createDataFrame(summary_data)

print("Detalle de tablas Silver:")
print("="*80)
display(df_summary)

total_records_individual = sum([item['Registros'] for item in summary_data[:-1]])

print("\n" + "="*80)
print("ESTADÍSTICAS SILVER")
print("="*80)
print(f"Tablas individuales:       5")
print(f"Registros individuales:    {total_records_individual:,}")
print(f"\nTabla consolidada:         {SILVER_SCHEMA}.clientes_consolidado")
print(f"Clientes con vista 360°:   {df_consolidado.count():,}")
print(f"Columnas en consolidado:   {len(df_consolidado.columns)}")
print(f"\nEsquema Silver:            {SILVER_SCHEMA}")
print(f"Optimización:              OPTIMIZE + ZORDER (nit)")
print("="*80)

print("\n✅ TRANSFORMACIÓN SILVER COMPLETADA EXITOSAMENTE")
print("\n📍 Siguiente paso: Ejecutar notebook 03_gold_feature_engineering")
print("   Comando: %run ./03_gold_feature_engineering\n")